In [3]:
import cv2
import os
import dlib
import glob as gb
import numpy as np
from matplotlib import pyplot as plt

# Load dlib's face detector and shape predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(r"C:\Users\Mohamed_Abdeldayem\Downloads\shape_predictor_68_face_landmarks.dat")
        
def facial_Landmarks(image,index):
    image = cv2.imread(image)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Detect faces
    faces = detector(gray)

    # Function to rotate the image
    def rotate_image(image, angle, center=None, scale=1.0):
        (h, w) = image.shape[:2]
        if center is None:
            center = (w // 2, h // 2)
        matrix = cv2.getRotationMatrix2D(center, angle, scale)
        rotated = cv2.warpAffine(image, matrix, (w, h))
        return rotated

    # Process the first detected face
    for face in faces:
        # Get the landmarks
        landmarks = predictor(gray, face)

        # Extract landmark points as (x, y)
        points = [(landmarks.part(n).x, landmarks.part(n).y) for n in range(68)]
        points = np.array(points, dtype=np.int32)

        # Calculate the angle between the eyes
        left_eye = np.mean(points[36:42], axis=0)  # Average of left eye landmarks
        right_eye = np.mean(points[42:48], axis=0)  # Average of right eye landmarks
        dx = right_eye[0] - left_eye[0]
        dy = right_eye[1] - left_eye[1]
        angle = np.degrees(np.arctan2(dy, dx))  # Angle in degrees

        # Rotate the image to align the eyes horizontally
        rotated_image = rotate_image(image, angle, center=tuple(left_eye))

        # Update the gray image after rotation
        rotated_gray = cv2.cvtColor(rotated_image, cv2.COLOR_BGR2GRAY)

        # Re-detect the face and landmarks on the rotated image
        faces_rotated = detector(rotated_gray)
        for face_rotated in faces_rotated:
            landmarks_rotated = predictor(rotated_gray, face_rotated)
            points_rotated = [(landmarks_rotated.part(n).x, landmarks_rotated.part(n).y) for n in range(68)]
            points_rotated = np.array(points_rotated, dtype=np.int32)

            # Define the region of interest
            x_min = np.min(points_rotated[:, 0])
            y_min = np.min(points_rotated[:, 1])
            x_max = np.max(points_rotated[:, 0])
            y_max = np.max(points_rotated[:, 1])

            # Add padding
            padding = 20
            x_min = max(0, x_min - padding)
            y_min = max(0, y_min - padding)
            x_max = min(rotated_image.shape[1], x_max + padding)
            y_max = min(rotated_image.shape[0], y_max + padding)

            # Crop the rotated image
            cropped_face = rotated_image[y_min:y_max, x_min:x_max]

            # Convert image to RGB for matplotlib
            image_rgb = cv2.cvtColor(cropped_face, cv2.COLOR_BGR2RGB)
            plt.imsave(output_dir+file[index:], image_rgb)

        break  # Process only the first face
        
# Read The Data For Training set
dataPath = 'D:/AffectNet/dataset/train_set/' 
for folder in  os.listdir(dataPath) :
    new_dataPath = dataPath+folder
    output_dir = 'D:/AffectNet/train_set/'+folder
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    files = gb.glob(pathname= str(new_dataPath +'/*.*g'))
    for file in files:
        facial_Landmarks(file,len(new_dataPath)+1)

ModuleNotFoundError: No module named 'cv2'